# **NCF Model**

Creates an Explicit NCF Model

In [1]:
import pandas as pd
import numpy as np
import kagglehub
from pathlib import Path
from tqdm.notebook import tqdm
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, ndcg_score

### **Load Data and Train-Test Split**

##### Load Data

In [2]:
# Download data if it isn't downloaded already

if not Path("books_db/ratings.csv").exists():
    kagglehub.dataset_download("zygmunt/goodbooks-10k", output_dir="books_db")
else:
    print("File already exists")

File already exists


In [3]:
# Read in csvs
file_path = "books_db"
ratings_df = pd.read_csv(f"{file_path}/ratings.csv")
books_df = pd.read_csv(f"{file_path}/books.csv")
display(ratings_df.head())
display(ratings_df.shape)

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4


(981756, 3)

##### Train-Test Split

In [4]:
# This split is kept consistent across models
rating_counts = ratings_df['user_id'].value_counts()
eligible_users = rating_counts[rating_counts >= 10].index
ratings_eligible = ratings_df[ratings_df['user_id'].isin(eligible_users)].copy()

def split_group_three_way(group, train_frac=0.6, val_frac=0.2, seed=42):
    train = group.sample(frac=train_frac, random_state=seed)
    remaining = group.drop(train.index)
    val = remaining.sample(frac=val_frac / (1 - train_frac), random_state=seed)
    test = remaining.drop(val.index)
    return train, val, test

train_parts = []
val_parts = []
test_parts = []

for user_id, group in ratings_eligible.groupby('user_id'):
    train, val, test = split_group_three_way(group)
    train_parts.append(train)
    val_parts.append(val)
    test_parts.append(test)

ratings_train = pd.concat(train_parts).reset_index(drop=True)
ratings_val = pd.concat(val_parts).reset_index(drop=True)
ratings_test = pd.concat(test_parts).reset_index(drop=True)

In [5]:
train_data = ratings_train
val_data = ratings_val
test_data = ratings_test

In [6]:
train_data

,book_id,user_id,rating
0,1199,7,4
1,3246,7,4
2,1646,7,3
3,585,7,4
4,4459,7,4
...,...,...,...
514848,8609,53424,4
514849,7833,53424,4
514850,8213,53424,4
514851,4214,53424,5


In [7]:
val_data

,book_id,user_id,rating
0,8320,7,5
1,4138,7,3
2,6445,7,4
3,4608,7,3
4,2470,7,3
...,...,...,...
170705,7667,53422,4
170706,4071,53422,4
170707,4483,53424,5
170708,5301,53424,5


In [8]:
test_data

,book_id,user_id,rating
0,956,7,5
1,1801,7,5
2,1923,7,4
3,2189,7,3
4,2325,7,4
...,...,...,...
171970,5811,53422,4
171971,8757,53422,5
171972,7212,53424,4
171973,7503,53424,4


### **Pre-processing**

In [9]:
# Check that all books in ratings_df are also in books_df
missing_books = ratings_df[~ratings_df['book_id'].isin(books_df['book_id'])]
print("Missing books count in books_df", missing_books['book_id'].nunique())

Missing books count in books_df 9188


In [10]:
# For user embedding lookup, check sizes
print("Unique Users:", ratings_df['user_id'].nunique())
print("Max User ID:", ratings_df['user_id'].max())
print("Unique Books:", books_df['book_id'].nunique())
print("Max Book ID:", books_df['book_id'].max())

Unique Users: 53424
Max User ID: 53424
Unique Books: 10000
Max Book ID: 33288638


In [11]:
# create new book column for embedding later (since max Book ID is large, it will make the lookup table too large)
def id_converter(unique_books):
    id_dictionary = {}
    for new_id, original_id in enumerate(unique_books):
        id_dictionary[original_id] = new_id
    return id_dictionary

# all_book_ids = pd.concat([books_df['book_id'], train_data['book_id'], test_data['book_id']])
all_book_ids = pd.concat([books_df['book_id'], ratings_df['book_id']])
unique_books = all_book_ids.unique()
id_dictionary = id_converter(unique_books)

train_data['lookup_book_id'] = train_data['book_id'].map(id_dictionary)
val_data['lookup_book_id'] = val_data['book_id'].map(id_dictionary)
test_data['lookup_book_id'] = test_data['book_id'].map(id_dictionary)
books_df['lookup_book_id'] = books_df['book_id'].map(id_dictionary)
ratings_df['lookup_book_id'] = ratings_df['book_id'].map(id_dictionary)

In [12]:
# Check book IDs to make sure there is the right amount
print("Unique Lookup Books:", books_df['lookup_book_id'].nunique())
print("Max Lookup Book ID:", books_df['lookup_book_id'].max())

Unique Lookup Books: 10000
Max Lookup Book ID: 9999


In [13]:
# just make sure there are no missing values
print("Missing books in train:", train_data['lookup_book_id'].isna().sum())
print("Missing books in val", val_data['lookup_book_id'].isna().sum())
print("Missing books in test", test_data['lookup_book_id'].isna().sum())

Missing books in train: 0
Missing books in val 0
Missing books in test 0


In [14]:
# get the total pool of books and check min and max ID
all_unique_books = pd.concat([books_df['lookup_book_id'], ratings_df['lookup_book_id']]).unique()
all_books_pool = np.array(all_unique_books)
print(f"Total unique books in the pool: {len(all_books_pool)}")

print("Contains NaN?", pd.isna(all_books_pool).any())

print("Min ID:", np.nanmin(all_books_pool))
print("Max ID:", np.nanmax(all_books_pool))

Total unique books in the pool: 19188
Contains NaN? False
Min ID: 0
Max ID: 19187


In [15]:
user_interacted_items = ratings_df.groupby('user_id')['lookup_book_id'].apply(set).to_dict()

### **Create Model**

In [16]:
# Use GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [17]:
class NCF_Explicit(nn.Module):
    def __init__(self, user_table_size, book_table_size, embedding_dim=32, hidden_dim1=64, hidden_dim2=32, dropout_rate=0.2):
        super().__init__()

        # Embeddings
        self.user_embedding = nn.Embedding(num_embeddings=user_table_size, embedding_dim=embedding_dim)
        self.book_embedding = nn.Embedding(num_embeddings=book_table_size, embedding_dim=embedding_dim)

        # MLP (3-4 hidden layers is standard)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, hidden_dim1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim2, 1),
        )

    def forward(self, user_input, book_input):
        user_vector = self.user_embedding(user_input)
        book_vector = self.book_embedding(book_input)

        combined_vector = torch.cat([user_vector, book_vector], dim=1)

        prediction = self.mlp(combined_vector)

        return prediction.squeeze()

In [18]:
# Training Loop
def train_model(model, dataloader, criterion, optimizer, epochs=5):
    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for users, books, targets in dataloader:
            users = users.to(device)
            books = books.to(device)
            targets = targets.to(device)
            optimizer.zero_grad()
            predictions = model(users, books)
            loss = criterion(predictions, targets.float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch + 1}: Loss {total_loss/len(dataloader):.4f}")

In [19]:
# For embedding lookup table size. It will be the entire pool of books. There were books in rating_df not in books_df and vis versa. All books were added.
user_table_size = ratings_df['user_id'].max() + 1
book_table_size = len(id_dictionary)

In [20]:
# DataLoader (Explicit Train)
users_train = torch.tensor(train_data['user_id'].values, dtype=torch.long)
books_train = torch.tensor(train_data['lookup_book_id'].values, dtype=torch.long)
ratings_train = torch.tensor(train_data['rating'].values, dtype=torch.float32)

train_dataset_exp = TensorDataset(users_train, books_train, ratings_train)
train_dataloader_exp = DataLoader(train_dataset_exp, batch_size=1024, shuffle=True)

In [ ]:
# # Evaluation(Explicit Val) - without decoys
# def explicit_eval(NCF_model_explicit, data, dataset_name="Validation", k=5):
#     """Evaluates the RMSE and MAE of the model"""
#     users_eval = torch.tensor(data['user_id'].values, dtype=torch.long)
#     books_eval = torch.tensor(data['lookup_book_id'].values, dtype=torch.long)
#     ratings_eval = torch.tensor(data['rating'].values, dtype=torch.float32)

#     eval_dataset_exp = TensorDataset(users_eval, books_eval, ratings_eval)
#     eval_dataloader_exp = DataLoader(eval_dataset_exp, batch_size=1024, shuffle=False)

#     NCF_model_explicit.eval()

#     all_predictions = []
#     all_targets = []

#     # Predict
#     with torch.no_grad():
#         for users, books, target_ratings in eval_dataloader_exp:
#             predicted_ratings = NCF_model_explicit(users, books).squeeze()

#             all_predictions.extend(predicted_ratings.numpy())
#             all_targets.extend(target_ratings.numpy())

#     all_predictions = np.array(all_predictions)
#     all_targets = np.array(all_targets)

#     mse = mean_squared_error(all_targets, all_predictions)
#     rmse = np.sqrt(mse)
#     mae = mean_absolute_error(all_targets, all_predictions)

#     results_df = pd.DataFrame({
#         'user_id': data['user_id'].values,
#         'true_rating': all_targets,
#         'pred_rating': all_predictions
#     })

#     user_ndcg_scores = []
#     for user, group in results_df.groupby('user_id'):
#         y_true = group['true_rating'].values.reshape(1, -1)
#         y_score = group['pred_rating'].values.reshape(1, -1)

#         score = ndcg_score(y_true, y_score, k=k)
#         user_ndcg_scores.append(score)

#     mean_ndcg = np.mean(user_ndcg_scores)

#     print(f"Explicit Model {dataset_name} RMSE: {rmse:.4f}")
#     print(f"Explicit Model {dataset_name} MAE: {mae:.4f}")
#     print(f"Explicit Model {dataset_name} NDCG@{k}: {mean_ndcg}")

In [21]:
# Evaluation(Explicit Val) - with decoys
def explicit_eval(NCF_model_explicit, data, all_books_pool, user_interacted_items, dataset_name="Validation", k=5, num_negatives=100):
    """Evaluates the RMSE and MAE of the model"""
    users_eval = torch.tensor(data['user_id'].values, dtype=torch.long)
    books_eval = torch.tensor(data['lookup_book_id'].values, dtype=torch.long)
    ratings_eval = torch.tensor(data['rating'].values, dtype=torch.float32)

    eval_dataset_exp = TensorDataset(users_eval, books_eval, ratings_eval)
    eval_dataloader_exp = DataLoader(eval_dataset_exp, batch_size=1024, shuffle=False)

    NCF_model_explicit.eval()

    all_predictions = []
    all_targets = []

    # Predict
    with torch.no_grad():
        for users, books, target_ratings in eval_dataloader_exp:
            predicted_ratings = NCF_model_explicit(users, books).squeeze()

            all_predictions.extend(predicted_ratings.numpy())
            all_targets.extend(target_ratings.numpy())

    all_predictions = np.array(all_predictions)
    all_targets = np.array(all_targets)

    mse = mean_squared_error(all_targets, all_predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(all_targets, all_predictions)

    results_df = pd.DataFrame({
        'user_id': data['user_id'].values,
        'true_rating': all_targets,
        'pred_rating': all_predictions
    })

    user_ndcg_scores = []

    for user, group in results_df.groupby('user_id'):
        eval_true_ratings = group['true_rating'].values
        eval_pred_ratings = group['pred_rating'].values

        # getting all items a user rated
        interacted = user_interacted_items.get(user, set())

        # getting the pool of never rated items
        valid_negatives = np.setdiff1d(all_books_pool, list(interacted))

        samples_negatives = np.random.choice(valid_negatives, size=num_negatives, replace=False)

        user_tensor = torch.tensor([user] * num_negatives, dtype=torch.long)
        neg_books_tensor = torch.tensor(samples_negatives, dtype=torch.long)

        with torch.no_grad():
            neg_pred_ratings = NCF_model_explicit(user_tensor, neg_books_tensor).view(-1).numpy()

        # Combine evaluation and negative books, assigning a true rating of zero to negatives
        combined_true = np.concatenate([eval_true_ratings, np.zeros(num_negatives)])
        combined_pred = np.concatenate([eval_pred_ratings, neg_pred_ratings])

        y_true = combined_true.reshape(1, -1)
        y_score = combined_pred.reshape(1, -1)

        score = ndcg_score(y_true, y_score, k=k)
        user_ndcg_scores.append(score)

    mean_ndcg = np.mean(user_ndcg_scores)

    print(f"Explicit Model {dataset_name} RMSE: {rmse:.4f}")
    print(f"Explicit Model {dataset_name} MAE: {mae:.4f}")
    print(f"Explicit Model {dataset_name} NDCG@{k}: {mean_ndcg:.4f}")

In [22]:
@dataclass
class TrainConfig:
    embedding_dim: int
    hidden_dim1: int
    hidden_dim2: int
    lr: float
    epochs: int
    dropout: float

##### Train Explicit

In [27]:
train_configs = [
    TrainConfig(embedding_dim=32, hidden_dim1=32, hidden_dim2=16, lr=0.001, epochs=20, dropout=0.2),
    TrainConfig(embedding_dim=64, hidden_dim1=64, hidden_dim2=32, lr=0.001, epochs=15, dropout=0.2),
    TrainConfig(embedding_dim=32, hidden_dim1=32, hidden_dim2=16, lr=0.001, epochs=10, dropout=0.1),
]

for i, config in enumerate(train_configs, start=1):
    print(f"Training Explicit NCF Model {i} - {config}")
    NCF_model_explicit = NCF_Explicit(user_table_size, book_table_size, config.embedding_dim, config.hidden_dim1, config.hidden_dim2, config.dropout).to(device)
    optimizer_explicit = optim.Adam(NCF_model_explicit.parameters(), config.lr)
    train_model(NCF_model_explicit, train_dataloader_exp, nn.MSELoss(), optimizer_explicit, config.epochs)
    explicit_eval(NCF_model_explicit, val_data, all_books_pool, user_interacted_items, "Validation", k=5, num_negatives=100)

Training Explicit NCF Model 1 - TrainConfig(embedding_dim=32, hidden_dim1=32, hidden_dim2=16, lr=0.001, epochs=20, dropout=0.2)
Epoch 1: Loss 2.7075
Epoch 2: Loss 1.4067
Epoch 3: Loss 1.2650
Epoch 4: Loss 1.1791
Epoch 5: Loss 1.1285
Epoch 6: Loss 1.0711
Epoch 7: Loss 1.0104
Epoch 8: Loss 0.9583
Epoch 9: Loss 0.8991
Epoch 10: Loss 0.8532
Epoch 11: Loss 0.8167
Epoch 12: Loss 0.7845
Epoch 13: Loss 0.7544
Epoch 14: Loss 0.7276
Epoch 15: Loss 0.7092
Epoch 16: Loss 0.6921
Epoch 17: Loss 0.6805
Epoch 18: Loss 0.6712
Epoch 19: Loss 0.6651
Epoch 20: Loss 0.6586
Explicit Model Validation RMSE: 0.8437
Explicit Model Validation MAE: 0.6605
Explicit Model Validation NDCG@5: 0.0797
Training Explicit NCF Model 2 - TrainConfig(embedding_dim=64, hidden_dim1=64, hidden_dim2=32, lr=0.001, epochs=15, dropout=0.2)
Epoch 1: Loss 2.2608
Epoch 2: Loss 1.2555
Epoch 3: Loss 1.1290
Epoch 4: Loss 1.0498
Epoch 5: Loss 0.9913
Epoch 6: Loss 0.9334
Epoch 7: Loss 0.8884
Epoch 8: Loss 0.8498
Epoch 9: Loss 0.8166
Epoch 

In [28]:
# Create explicit NCF model
optimal_config = TrainConfig(embedding_dim=64, hidden_dim1=64, hidden_dim2=32, lr=0.001, epochs=15, dropout=0.2)
NCF_model_explicit = NCF_Explicit(user_table_size, book_table_size, optimal_config.embedding_dim, optimal_config.hidden_dim1, optimal_config.hidden_dim2, optimal_config.dropout).to(device)
optimizer_explicit = optim.Adam(NCF_model_explicit.parameters(), optimal_config.lr)
train_model(NCF_model_explicit, train_dataloader_exp, nn.MSELoss(), optimizer_explicit, optimal_config.epochs)

Epoch 1: Loss 2.7178
Epoch 2: Loss 1.3231
Epoch 3: Loss 1.2129
Epoch 4: Loss 1.1277
Epoch 5: Loss 1.0588
Epoch 6: Loss 1.0036
Epoch 7: Loss 0.9521
Epoch 8: Loss 0.9038
Epoch 9: Loss 0.8618
Epoch 10: Loss 0.8220
Epoch 11: Loss 0.7885
Epoch 12: Loss 0.7578
Epoch 13: Loss 0.7279
Epoch 14: Loss 0.7014
Epoch 15: Loss 0.6804


In [29]:
explicit_eval(NCF_model_explicit, val_data, all_books_pool, user_interacted_items, "Validation", k=5, num_negatives=100)

Explicit Model Validation RMSE: 0.8467
Explicit Model Validation MAE: 0.6633
Explicit Model Validation NDCG@5: 0.0890


### **Evaluation**

##### RMSE, MAE, NDCG@K Test

In [30]:
explicit_eval(NCF_model_explicit, test_data, all_books_pool, user_interacted_items, "Test", k=5, num_negatives=100)

Explicit Model Test RMSE: 0.8457
Explicit Model Test MAE: 0.6621
Explicit Model Test NDCG@5: 0.0885
